In [1]:
import pandas as pd
import altair as alt
import os

DATA_PATH  = os.path.join("2025-us-b-visa-denial-rate.csv")

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

There is 199 countries

In [3]:
df.shape

(199, 2)

In [4]:
df.head()

,NATIONALITY,FY25 ADJUSTED REFUSAL RATE
0,AFGHANISTAN,63.25%
1,ALBANIA,37.32%
2,ALGERIA,51.23%
3,ANDORRA,9.09%
4,ANGOLA,54.33%


In [5]:
df.rename(columns={
    'FY25 ADJUSTED REFUSAL RATE':    'Refusal_Rate',
    'NATIONALITY':                   'Nationality'
}, inplace=True)

In [6]:
df.head()

,Nationality,Refusal_Rate
0,AFGHANISTAN,63.25%
1,ALBANIA,37.32%
2,ALGERIA,51.23%
3,ANDORRA,9.09%
4,ANGOLA,54.33%


In [7]:
df['Refusal_Rate'] = df['Refusal_Rate'].str.rstrip('%').astype(float) / 100.0

In [8]:
df.head()

,Nationality,Refusal_Rate
0,AFGHANISTAN,0.6325
1,ALBANIA,0.3732
2,ALGERIA,0.5123
3,ANDORRA,0.0909
4,ANGOLA,0.5433


In [9]:
df.sort_values('Refusal_Rate', ascending=False).head(50)


,Nationality,Refusal_Rate
59,FEDERATED STATES OF MICRONESIA,1.0000
98,LAOS,0.8435
162,SOMALIA,0.8352
137,PALAU,0.8000
164,SOUTH SUDAN,0.7609
64,"GAMBIA, THE",0.7529
73,GUINEA - BISSAU,0.7517
154,SENEGAL,0.7396
102,LIBERIA,0.7301
180,TOGO,0.7202


In [10]:
df.sort_values('Refusal_Rate').head(50)


,Nationality,Refusal_Rate
104,LIECHTENSTEIN,0.0000
193,VATICAN CITY,0.0000
119,MONACO,0.0000
189,UNITED ARAB EMIRATES,0.0217
45,CYPRUS,0.0255
190,URUGUAY,0.0259
135,OMAN,0.0314
92,KIRIBATI,0.0450
186,TUVALU,0.0455
25,BULGARIA,0.0511


#### Step 2 - standardize country naming

In [1]:
%pip install country_converter

Note: you may need to restart the kernel to use updated packages.


In [20]:
import country_converter as coco

In [21]:
df['ISO3']       = coco.convert(names=df['Nationality'], to='ISO3')
df['name_short'] = coco.convert(names=df['Nationality'], to='name_short')
df.head()

*NON-NATIONALITY BASED ISSUANCES not found in regex
*NON-NATIONALITY BASED ISSUANCES not found in regex


,Nationality,Refusal_Rate,ISO3,name_short
0,AFGHANISTAN,0.6325,AFG,Afghanistan
1,ALBANIA,0.3732,ALB,Albania
2,ALGERIA,0.5123,DZA,Algeria
3,ANDORRA,0.0909,AND,Andorra
4,ANGOLA,0.5433,AGO,Angola


In [24]:
# coco flags anything it can't resolve as 'not found'
# For this dataset the ONLY unresolved row is '*NON-NATIONALITY BASED ISSUANCES', which isn't a country.
df[df['ISO3'] == 'not found']

,Nationality,Refusal_Rate,ISO3,name_short


In [25]:
#drop this row 

df = df[df['Nationality'] != '*NON-NATIONALITY BASED ISSUANCES'].reset_index(drop=True)


In [26]:
# Drop the non-country row, then confirm everything else resolved (should be 0 remaining).
print("rows:", len(df), "| still not found:", (df['ISO3'] == 'not found').sum())


rows: 198 | still not found: 0


In [27]:
df[['Nationality', 'name_short', 'ISO3', 'Refusal_Rate']].head(10)

,Nationality,name_short,ISO3,Refusal_Rate
0,AFGHANISTAN,Afghanistan,AFG,0.6325
1,ALBANIA,Albania,ALB,0.3732
2,ALGERIA,Algeria,DZA,0.5123
3,ANDORRA,Andorra,AND,0.0909
4,ANGOLA,Angola,AGO,0.5433
5,ANTIGUA AND BARBUDA,Antigua and Barbuda,ATG,0.3889
6,ARGENTINA,Argentina,ARG,0.0747
7,ARMENIA,Armenia,ARM,0.5637
8,AUSTRALIA,Australia,AUS,0.2293
9,AUSTRIA,Austria,AUT,0.0892


In [29]:
df['iso_num'] = coco.convert(df['Nationality'], to='ISOnumeric')
df['iso_num'] = df['iso_num'].astype(str).str.zfill(3)

In [30]:
df.head()

,Nationality,Refusal_Rate,ISO3,name_short,iso_num
0,AFGHANISTAN,0.6325,AFG,Afghanistan,004
1,ALBANIA,0.3732,ALB,Albania,008
2,ALGERIA,0.5123,DZA,Algeria,012
3,ANDORRA,0.0909,AND,Andorra,020
4,ANGOLA,0.5433,AGO,Angola,024


In [31]:
df.to_csv('cleaned_refusal.csv', index=False)
